# House Rent Prediction — Regression Pipeline

This notebook formulates **House_Rent_Dataset.csv** as a **Regression** problem predicting the continuous target variable **`Rent`** (monthly house rent in INR), adapted from the reference `Regression Models.ipynb` workflow.


## 1. Importing Libraries

Load the required data science, visualization, preprocessing, regression model, and evaluation metric libraries, including optional gradient boosting libraries (`xgboost`, `catboost`, `lightgbm`) with safe import fallbacks.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    BaggingRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor
)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False


## 2. Exploratory Data Analysis (EDA)

Load `House_Rent_Dataset.csv` and inspect its shape, first rows, structure, missing values, and summary statistics.


In [ ]:
df = pd.read_csv('House_Rent_Dataset.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
df.describe(include='all')


### 2.1 City, BHK, and Furnishing Breakdown

Examine listing counts and median/mean rent across cities, BHK configurations, and furnishing levels.


In [ ]:
city_summary = df.groupby('City').agg(
    Listings=('Rent', 'count'),
    Rent_Mean=('Rent', 'mean'),
    Rent_Median=('Rent', 'median'),
    Rent_Min=('Rent', 'min'),
    Rent_Max=('Rent', 'max'),
    Size_Median=('Size', 'median')
).sort_values('Rent_Median', ascending=False)
city_summary


In [ ]:
print('BHK counts:')
print(df['BHK'].value_counts().sort_index())
print('\nFurnishing Status counts:')
print(df['Furnishing Status'].value_counts())
print('\nArea Type counts:')
print(df['Area Type'].value_counts(dropna=False))


## 3. Data Visualization

Visualize rent distribution, median rent by city, BHK frequency, and rent versus property size.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
city_med = df.groupby('City')['Rent'].median().sort_values(ascending=True)
axes[0].barh(city_med.index, city_med.values)
axes[0].set_title('Median Monthly Rent by City')
axes[0].set_xlabel('Median Rent (INR)')
bhk_cnt = df['BHK'].value_counts().sort_index()
axes[1].bar(bhk_cnt.index.astype(str), bhk_cnt.values)
axes[1].set_title('Number of Listings by BHK')
axes[1].set_xlabel('BHK')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Furnishing Status', y='Rent', ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('Rent Distribution by Furnishing Status (Log Scale)')
sns.scatterplot(data=df, x='Size', y='Rent', hue='City', alpha=0.5, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Size vs. Rent by City (Log Scale)')
plt.tight_layout()
plt.show()


## 4. Data Cleaning and Feature Engineering

Remove rows with missing `Rent` target values, parse `Floor` into numeric `Current_Floor` and `Total_Floors`, and extract calendar components (`Posted_Month`, `Posted_Day`, `Posted_DayOfWeek`) from `Posted On`.


In [ ]:
data = df.dropna(subset=['Rent']).copy()
def parse_floor(val):
    s = str(val).strip()
    parts = s.split(' out of ')
    cur_raw = parts[0].strip().lower()
    tot_raw = parts[1].strip() if len(parts) > 1 else '0'
    if cur_raw == 'ground':
        cur = 0
    elif cur_raw == 'upper basement':
        cur = -1
    elif cur_raw == 'lower basement':
        cur = -2
    else:
        try:
            cur = int(cur_raw)
        except ValueError:
            cur = 0
    try:
        tot = int(tot_raw)
    except ValueError:
        tot = 0
    return pd.Series([cur, tot])
data[['Current_Floor', 'Total_Floors']] = data['Floor'].apply(parse_floor)
posted_dt = pd.to_datetime(data['Posted On'])
data['Posted_Month'] = posted_dt.dt.month
data['Posted_Day'] = posted_dt.dt.day
data['Posted_DayOfWeek'] = posted_dt.dt.dayofweek
data = data.drop(columns=['Posted On', 'Floor'])
print('Cleaned dataset shape:', data.shape)


## 5. Train/Test Split, Imputation, Encoding, and Scaling

Separate features (`X`) and target (`y = data[['Rent']]`), split 80/20 with `random_state=100`, impute missing values and encode categorical features strictly on training data to prevent leakage, and scale features using `MinMaxScaler(feature_range=(0, 1))`.


In [ ]:
X = data.drop('Rent', axis=1)
y = data[['Rent']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=100)
print('X_train =', X_train.shape, 'y_train =', y_train.shape)
print('X_test =', X_test.shape, 'y_test =', y_test.shape)


In [ ]:
num_cols = ['BHK', 'Size', 'Bathroom', 'Current_Floor', 'Total_Floors', 'Posted_Month', 'Posted_Day', 'Posted_DayOfWeek']
cat_cols = ['Area Type', 'Area Locality', 'City', 'Furnishing Status', 'Tenant Preferred', 'Point of Contact']
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])
my_scaler = MinMaxScaler(feature_range=(0, 1))
X_train = my_scaler.fit_transform(X_train)
X_test = my_scaler.transform(X_test)


## 6. Evaluation Metrics Definition

Define the `regression_metrics` helper computing `MAE`, `MSE`, `RMSE`, and `R2` (`r2_score`), matching the reference regression notebook.


In [ ]:
def regression_metrics(model, X_data, y_actual):
    y_pred = model.predict(X_data)
    MAE = mean_absolute_error(y_actual, y_pred)
    MSE = mean_squared_error(y_actual, y_pred)
    RMSE = np.sqrt(MSE)
    R2 = r2_score(y_actual, y_pred)
    return MAE, MSE, RMSE, R2


## 7. Individual Regression Models Training and Evaluation

Train all reference regression models (`Linear Regression`, `Lasso`, `Ridge`, `KNN`, `SVM`, `Decision Tree`, `Random Forest`, `ExtraTrees`, `Bagging`, `AdaBoost`, `GradientBoosting`, `HistGradientBoosting`, plus `XGBoost`, `CatBoost`, and `LightGBM` when available), record train and test metrics, and plot actual versus predicted rent for the first 20 test samples.


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'KNN': KNeighborsRegressor(n_neighbors=5, metric='minkowski', p=1),
    'SVM': SVR(kernel='linear', C=100, gamma=1),
    'Decision Tree': DecisionTreeRegressor(criterion='squared_error', max_depth=10),
    'Random Forest': RandomForestRegressor(n_estimators=50, criterion='squared_error', max_depth=10),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Bagging': BaggingRegressor(estimator=DecisionTreeRegressor(criterion='squared_error', max_depth=10), n_estimators=50, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=15, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=100, learning_rate=0.1, random_state=42)
}
if HAS_XGB:
    models['XGBoost'] = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
if HAS_CAT:
    models['CatBoost'] = CatBoostRegressor(verbose=0, iterations=100, learning_rate=0.1, random_state=42)
if HAS_LGBM:
    models['LightGBM'] = LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)


In [ ]:
y_train_1d = y_train.values.ravel()
y_test_1d = y_test.values.ravel()
errors = {}
for model_name, model in tqdm(models.items()):
    model.fit(X_train, y_train_1d)
    MAE_Train, MSE_Train, RMSE_Train, R_Train = regression_metrics(model, X_train, y_train_1d)
    MAE_Test, MSE_Test, RMSE_Test, R_Test = regression_metrics(model, X_test, y_test_1d)
    errors[model_name] = {
        'MAE Train': MAE_Train,
        'MAE Test': MAE_Test,
        'MSE Train': MSE_Train,
        'MSE Test': MSE_Test,
        'RMSE Train': RMSE_Train,
        'RMSE Test': RMSE_Test,
        'R2 Train': R_Train,
        'R2 Test': R_Test
    }
    y_test_plot = y_test_1d[:20]
    y_pred_plot = model.predict(X_test[:20])
    plt.figure(figsize=(10, 5))
    plt.plot(range(20), y_test_plot, label='Actual Rent', marker='o')
    plt.plot(range(20), y_pred_plot, label='Predicted Rent', marker='x')
    plt.xlabel('Sample Index')
    plt.ylabel('Rent Amount')
    plt.title('Actual vs Predicted Rent (First 20 Samples) For ' + model_name)
    plt.legend()
    plt.grid()
    plt.show()


## 8. Voting Regression

Fit equal-weight and custom-weighted `VotingRegressor` ensembles (`Voting`, `Simple Voting`, and `Weighted Voting`) and record their metrics.


In [ ]:
clf1 = LinearRegression()
clf2 = RandomForestRegressor(n_estimators=50, random_state=1)
clf3 = DecisionTreeRegressor(random_state=1)
model1 = VotingRegressor(estimators=[('lr', clf1), ('rf', clf2), ('dt', clf3)])
model2 = VotingRegressor(estimators=[('lr', clf1), ('rf', clf2), ('dt', clf3)])
model3 = VotingRegressor(estimators=[('lr', clf1), ('rf', clf2), ('dt', clf3)], weights=[2, 1, 2])
model1.fit(X_train, y_train_1d)
model2.fit(X_train, y_train_1d)
model3.fit(X_train, y_train_1d)


In [ ]:
for v_name, v_model in [('Voting', model1), ('Simple Voting', model2), ('Weighted Voting', model3)]:
    MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(v_model, X_train, y_train_1d)
    MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(v_model, X_test, y_test_1d)
    errors[v_name] = {
        'MAE Train': MAE_Train,
        'MAE Test': MAE_Test,
        'MSE Train': MSE_Train,
        'MSE Test': MSE_Test,
        'RMSE Train': RMSE_Train,
        'RMSE Test': RMSE_Test,
        'R2 Train': R2_Train,
        'R2 Test': R2_Test
    }


## 9. Stacking Regression

Train a `StackingRegressor` combining tree and boosting base regressors with a meta-regressor (`passthrough=True`, `cv=5`) and record its metrics.


In [ ]:
base_regressors = [
    ('rf', RandomForestRegressor(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, random_state=42)),
    ('hgb', HistGradientBoostingRegressor(max_iter=50, learning_rate=0.1, random_state=42))
]
if HAS_CAT:
    base_regressors.append(('cat', CatBoostRegressor(iterations=50, learning_rate=0.1, verbose=0, random_state=42)))
if HAS_LGBM:
    base_regressors.append(('lgb', LGBMRegressor(n_estimators=50, learning_rate=0.1, random_state=42, verbose=-1)))
meta_regressor = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0) if HAS_XGB else GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
stacking_regressor = StackingRegressor(
    estimators=base_regressors,
    final_estimator=meta_regressor,
    cv=5,
    passthrough=True,
    n_jobs=-1
)
stacking_regressor.fit(X_train, y_train_1d)
MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(stacking_regressor, X_train, y_train_1d)
MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(stacking_regressor, X_test, y_test_1d)
errors['Stacking'] = {
    'MAE Train': MAE_Train,
    'MAE Test': MAE_Test,
    'MSE Train': MSE_Train,
    'MSE Test': MSE_Test,
    'RMSE Train': RMSE_Train,
    'RMSE Test': RMSE_Test,
    'R2 Train': R2_Train,
    'R2 Test': R2_Test
}


## 10. Model Comparison Table and Metric Visualizations

Compile all models into `errors_df` sorted by `R2 Test` descending and plot comparison charts for `R2`, `MAE`, `MSE`, and `RMSE`.


In [ ]:
errors_df = pd.DataFrame(errors).T.sort_values(by='R2 Test', ascending=False)
errors_df


In [ ]:
errors_df[['R2 Train', 'R2 Test']].plot.bar(figsize=(12, 6))
plt.title('R2 Score Comparison (Train vs Test)')
plt.ylabel('R2 Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
cols = [
    ['MAE Train', 'MAE Test'],
    ['MSE Train', 'MSE Test'],
    ['RMSE Train', 'RMSE Test'],
    ['R2 Train', 'R2 Test']
]
plt.figure(figsize=(20, 16))
for i, col in enumerate(cols):
    ax = plt.subplot(2, 2, i + 1)
    errors_df[col].plot.bar(ax=ax)
    ax.set_title(col[0].split(' ')[0] + ' Comparison')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 11. Final Conclusion

- **Problem Formulation:** `House_Rent_Dataset.csv` (4,746 rows, 12 columns) is a **Regression** problem predicting continuous monthly `Rent` across six Indian cities (`Mumbai`, `Delhi`, `Bangalore`, `Chennai`, `Hyderabad`, and `Kolkata`).
- **Market & Data Characteristics:** Rent is strongly right-skewed (`Median = 16,000.00 INR` vs. `Mean = 35,023.40 INR`, up to `3,500,000.00 INR`), led by `Mumbai` (`Median = 52,000.00 INR`), and scales with `BHK`, `Size`, and furnishing level.
- **Model Performance Summary:** Ensemble and gradient boosting methods (`Stacking`, `HistGradientBoosting`, and `GradientBoosting`) achieve the highest test `R2` and lowest test error (`MAE` and `RMSE`) compared to linear models (`Linear Regression`, `Lasso`, `Ridge`), `KNN`, `SVM`, and unpruned/single-tree models.


## 10.1 Regression Reports for All Regression Models

Print test MAE, MSE, RMSE, and R2 score across every evaluated regression model.


In [ ]:
all_models = dict(models)
all_models['Voting'] = model1
all_models['Simple Voting'] = model2
all_models['Weighted Voting'] = model3
all_models['Stacking'] = stacking_regressor
preds_dict = {name: model.predict(X_test) for name, model in all_models.items()}


In [ ]:
for name in preds_dict.keys():
    y_pred = preds_dict[name]
    mae = mean_absolute_error(y_test_1d, y_pred)
    mse = mean_squared_error(y_test_1d, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_1d, y_pred)
    print(f'=== {name} ===')
    print(f'MAE  : {mae:.4f}')
    print(f'MSE  : {mse:.4f}')
    print(f'RMSE : {rmse:.4f}')
    print(f'R2   : {r2:.6f}\n')
